# OOP: Classes, Inheritance & Dunder Protocols (5+ Years Interview Guide)
Exhaustive revision guide to class structures, @classmethod/@staticmethod, super() inheritance, and magic dunders (__str__, __repr__, __eq__, __getitem__, __add__) on raw_transactions.csv.

### Key 5-Year Interview Concepts Covered:
- **Class Structure & Methods**: Dedicated cell for `class`, `__init__`, `@classmethod`, and `@staticmethod`.
- **Inheritance & Polymorphism**: Dedicated cell for `class Child(Parent):` and `super()`.
- **Representation Magic Dunders**: Dedicated cell for `__str__` and `__repr__`.
- **Comparison & Container Magic Dunders**: Dedicated cell for `__eq__`, `__lt__`, `__getitem__`, and `__add__`.

This interactive revision guide uses `data/raw_transactions.csv` with individual dedicated cells per method.

In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import csv
import sys
import time
import os
import json
import re
import collections
from datetime import datetime, timedelta

csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
transactions = []
with open(csv_path, mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        transactions.append(row)

print(f"Python Version: {sys.version.split()[0]}")
print(f"Loaded {len(transactions)} transaction records from {csv_path}")

Python Version: 3.12.7
Loaded 15000 transaction records from data/raw_transactions.csv


### Class Structure, Constructors & Method Types
**Explanation**: `@classmethod` receives `cls` for alternative factory constructors. `@staticmethod` contains pure logic without `self` or `cls`.

**Syntax**: `class Transaction: @classmethod def from_dict(cls, d): ...`

In [2]:
class Transaction:
    category = 'Financial_Record'
    
    def __init__(self, tx_id: str, amount: float, is_fraud: bool):
        self.tx_id = tx_id
        self.amount = amount
        self.is_fraud = is_fraud
        
    @classmethod
    def from_dict(cls, row_dict):
        return cls(row_dict['transaction_id'], float(row_dict['transaction_amount']), bool(int(row_dict['is_fraud'])))
        
    @staticmethod
    def is_valid_currency(symbol):
        return symbol in ['USD', 'EUR', 'GBP']

tx_obj = Transaction.from_dict(transactions[0])
print(f'Constructed Instance: {tx_obj.tx_id}, Amount: ${tx_obj.amount}, Valid USD: {Transaction.is_valid_currency("USD")}')

Constructed Instance: TX110686, Amount: $1216.33, Valid USD: True


### Inheritance, `super()`, and Polymorphism
**Explanation**: `super().__init__()` delegates initialization to base classes through C3 Method Resolution Order (MRO).

**Syntax**: `class FraudAlertTransaction(Transaction): super().__init__(...)`

In [3]:
class FraudAlertTransaction(Transaction):
    def __init__(self, tx_id: str, amount: float, risk_score: float):
        super().__init__(tx_id, amount, is_fraud=True)
        self.risk_score = risk_score
        
    def get_summary(self):
        return f'ALERT: {self.tx_id} flagged with risk score {self.risk_score}'

alert_obj = FraudAlertTransaction('TX_ALERT_1', 1500.0, 0.95)
print(alert_obj.get_summary())
print('MRO:', [c.__name__ for c in FraudAlertTransaction.__mro__])

ALERT: TX_ALERT_1 flagged with risk score 0.95
MRO: ['FraudAlertTransaction', 'Transaction', 'object']


### Representation Protocols: `__repr__` vs `__str__`
**Explanation**: `__repr__` provides unambiguous technical representation for developers (`eval(repr(obj)) == obj`). `__str__` provides human-readable representation.

**Syntax**: `def __repr__(self): ...` / `def __str__(self): ...`

In [4]:
class TransactionRecord:
    def __init__(self, tx_id, amount):
        self.tx_id = tx_id
        self.amount = amount
    def __repr__(self):
        return f'TransactionRecord(tx_id={self.tx_id!r}, amount={self.amount!r})'
    def __str__(self):
        return f'Tx[{self.tx_id}]: ${self.amount:.2f}'

tx_rec = TransactionRecord(transactions[0]['transaction_id'], float(transactions[0]['transaction_amount']))
print('repr():', repr(tx_rec))
print('str():', str(tx_rec))

repr(): TransactionRecord(tx_id='TX110686', amount=1216.33)
str(): Tx[TX110686]: $1216.33


### Operator Overloading Protocols: `__eq__`, `__add__`, `__getitem__`
**Explanation**: Magic dunders implement Python protocols for equality testing, arithmetic addition, and dictionary/sequence indexing.

**Syntax**: `def __eq__(self, other): ...` / `def __add__(self, other): ...`

In [5]:
class TransactionBatch:
    def __init__(self, items):
        self.items = items
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        return self.items[idx]
    def __add__(self, other):
        return TransactionBatch(self.items + other.items)

batch1 = TransactionBatch([transactions[0], transactions[1]])
batch2 = TransactionBatch([transactions[2]])
combined = batch1 + batch2
print(f'Combined Batch Length: {len(combined)}, First item ID: {combined[0]["transaction_id"]}')

Combined Batch Length: 3, First item ID: TX110686


## Section: Senior Fintech Interview Scenarios (5+ Years Experience)

### Q1: Multiple Inheritance & C3 Linearization Algorithm (MRO)
**Explanation**: Explain how Python resolves diamond inheritance using C3 Linearization to guarantee monotonicity and local precedence order.

**Syntax**: `Class.__mro__`

In [6]:
class A: pass
class B(A): pass
class C(A): pass
class D(B, C): pass
print('Method Resolution Order for D:', [cls.__name__ for cls in D.__mro__])

Method Resolution Order for D: ['D', 'B', 'C', 'A', 'object']
